# Ingesta de relaciones película-idioma (incremental)

Lee la carpeta `movie_language` (JSON multilínea) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.movies_languages` con `MERGE` (llave `movie_id` + `language_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee la carpeta `movie_language` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_language_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_language")
    )

In [0]:
movie_language_schema = StructType([
    StructField("movieId", StringType()),
    StructField("languageId", IntegerType()),
    StructField("languageRoleId", IntegerType())
])

movie_language_df = (
    spark.read
        .schema(movie_language_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_language")
)

### Transformación de columnas

Renombra `movieId` y `languageId`, descarta `languageRoleId` y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
movie_language_df = (
    movie_language_df
        .withColumnsRenamed(
            {
                "movieId": "movie_id",
                "languageId": "language_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env),
                "file_date": lit(v_file_date)
            }
        )
).drop("languageRoleId")

movie_language_df = add_ingestion_date(movie_language_df)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.movies_languages` con la llave `movie_id` + `language_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "movies_languages", "file_date", v_file_date)

In [0]:
"""
(
    movie_language_df
        .write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable("movie_silver.movies_languages")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.language_id = src.language_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movie_language_df, schema_name = "movie_silver", table_name = "movies_languages", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")